In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from captum.attr import IntegratedGradients
from scipy.stats import pearsonr

# --------- 1. 단어 급수 예측 모델 (사용자 정의) ----------
# 예: 사용자 모델 로딩 또는 함수 정의
def word_model(word):
    return int(hash(word) % 6 + 1)  # 임시급수 (1~6)

# --------- 2. 문서 분류 모델 정의 (Bi-LSTM 기반 예시) ----------
class DocClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        emb = self.embedding(x)
        out, (hn, _) = self.lstm(emb)
        return self.fc(hn[-1])

# --------- 3. 입력 준비 ----------
def build_vocab(texts):
    vocab = {'[PAD]': 0, '[UNK]': 1}
    for text in texts:
        for tok in text.split():
            if tok not in vocab:
                vocab[tok] = len(vocab)
    return vocab

def tokenize(text, vocab):
    return [vocab.get(tok, vocab['[UNK]']) for tok in text.split()]

# --------- 4. 문서급 Integrated Gradients 분석 ----------
def explain_ig(model, input_tensor, label_idx):
    ig = IntegratedGradients(model)
    model.eval()

    def forward_func(x):
        return model(x)[..., label_idx]

    attributions, _ = ig.attribute(
        input_tensor.unsqueeze(0),
        n_steps=50,
        return_convergence_delta=False,
        target=label_idx
    )
    return attributions.squeeze(0).detach().numpy()

# --------- 5. 실행 파이프라인 ----------
def run_pipeline(data_csv_path):
    df = pd.read_csv(data_csv_path)  # text,label 컬럼 포함
    vocab = build_vocab(df['text'])
    model = DocClassifier(vocab_size=len(vocab), emb_dim=100, hidden_dim=64, num_classes=6)
    model.load_state_dict(torch.load("doc_model.pt"))  # 사전학습된 문서 모델 사용
    model.eval()

    avg_word_levels = []
    weighted_importance = []
    true_labels = []

    for idx, row in df.iterrows():
        text = row['text']
        label = int(row['label']) - 1  # 0-based
        tokens = text.split()

        # 단어급수 예측
        word_levels = [word_model(tok) for tok in tokens]
        avg_level = np.mean(word_levels)

        # 문서 임베딩 및 예측
        input_ids = torch.tensor(tokenize(text, vocab))
        pred_label = torch.argmax(model(input_ids.unsqueeze(0))).item()

        # IG 계산
        ig_scores = explain_ig(model, input_ids, label)

        # 단어급수 * IG
        min_len = min(len(word_levels), len(ig_scores))
        weighted_score = np.sum(np.array(word_levels[:min_len]) * ig_scores[:min_len])

        avg_word_levels.append(avg_level)
        weighted_importance.append(weighted_score)
        true_labels.append(label + 1)

    # --------- 6. 시각화 및 상관관계 분석 ----------
    fig, axs = plt.subplots(1, 2, figsize=(12, 5))

    axs[0].scatter(avg_word_levels, true_labels)
    axs[0].set_title("평균 단어급수 vs 실제 토픽급수")
    axs[0].set_xlabel("평균 단어급수")
    axs[0].set_ylabel("실제 토픽급수")

    axs[1].scatter(weighted_importance, true_labels)
    axs[1].set_title("단어급수 * IG vs 실제 토픽급수")
    axs[1].set_xlabel("급수 * 영향력")
    axs[1].set_ylabel("실제 토픽급수")

    plt.tight_layout()
    plt.show()

    # 상관관계 분석
    corr1 = pearsonr(avg_word_levels, true_labels)[0]
    corr2 = pearsonr(weighted_importance, true_labels)[0]
    print(f"상관계수 (평균 단어급수 vs 실제): {corr1:.3f}")
    print(f"상관계수 (급수*IG vs 실제): {corr2:.3f}")

# 실행
run_pipeline("documents.csv")